<a href="https://colab.research.google.com/github/amiralirh/Diabetic-Retinopathy-Detection/blob/main/DR_offline_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-image


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.restoration import wiener # For Wiener filter
from google.colab import drive

##  Directory Configuration and Data Transfer




When training deep learning models in cloud environments like Google Colab, reading thousands of high-resolution image files directly from networked storage (Google Drive) creates a severe I/O bottleneck. This data starvation causes the GPU to idle while waiting for batches to load, which drastically increases training time.

To optimize the data pipeline performance, this step configures our working directories and transfers the raw dataset directly to the local Colab disk before any preprocessing begins.

**Key Actions in this Cell:**
* **Path Definition:** Establishes clear, distinct directories for the source data on Google Drive, the raw local copy, and the eventual preprocessed output.
* **Local Transfer:** Utilizes the `rsync` utility for efficient, verifiable copying of the training images from the Drive mount to the local compute instance (`/content/aptos_local_data_raw`).

In [ ]:
# --- 2. Define Source and Destination Paths ---
SOURCE_DATA_DIR_ON_DRIVE = '/content/drive/MyDrive/Aptos'
SOURCE_TRAIN_IMAGES_DIR_ON_DRIVE = os.path.join(SOURCE_DATA_DIR_ON_DRIVE, 'train')

# Destination paths for storing copied local data
LOCAL_DATA_DIR_ON_COLAB = '/content/aptos_local_data_raw' # Raw data is copied here first
LOCAL_TRAIN_IMAGES_DIR = os.path.join(LOCAL_DATA_DIR_ON_COLAB, 'train')

# Destination paths for storing final preprocessed data
PREPROCESSED_DATA_DIR = '/content/aptos_preprocessed'
PREPROCESSED_TRAIN_IMAGES_DIR = os.path.join(PREPROCESSED_DATA_DIR, 'preprocessed_train_images')

# --- Copy raw data from Google Drive to Colab local storage ---
print(f"Copying raw data from Google Drive to {LOCAL_DATA_DIR_ON_COLAB}...")
os.makedirs(LOCAL_TRAIN_IMAGES_DIR, exist_ok=True)
# Note: LOCAL_TEST_IMAGES_DIR was missing in previous cell logic, ensure it is defined if needed
!rsync -avh "$SOURCE_TRAIN_IMAGES_DIR_ON_DRIVE/" "$LOCAL_TRAIN_IMAGES_DIR/"
print("Data copy to Colab local storage completed.")

در حال کپی کردن داده های خام از Google Drive به /content/aptos_local_data_raw...
sending incremental file list
./
000c1434d8d7.png
001639a390f0.png
0024cdab0c1e.png
002c21358ce6.png
005b95c28852.png
0083ee8054ee.png
0097f532ac9f.png
00a8624548a9.png
00b74780d31d.png
00cb6555d108.png
00cc2b75cddd.png
00e4ddff966a.png
00f6c1be5a33.png
0104b032c141.png
0124dffecf29.png
0125fbd2e791.png
012a242ac6ff.png
014508ccb9cb.png
0151781fe50b.png
0161338f53cc.png
0180bfa26c0b.png
0182152c50de.png
01b3aed3ed4c.png
01c7808d901d.png
01d9477b1171.png
01eb826f6467.png
01f7bb8be950.png
0212dd31f623.png
022f820027b8.png
0231642cf1c2.png
0232dfea7547.png
02358b47ea89.png
0243404e8a00.png
025a169a0bb0.png
02685f13cefd.png
026dcd9af143.png
02cd34a85b24.png
02da652c74b8.png
02dda30d3acf.png
0304bedad8fe.png
0318598cfd16.png
032d7b0b4bf6.png
033f2b43de6d.png
034cb07a550f.png
03676c71ed1b.png
0369f3efe69b.png
03747397839f.png
03a7f4a5786f.png
03b373718013.png
03c85870824c.png
03e25101e8e8.png
03fd50da928d.png
03

## Retinal Image Preprocessing Pipeline

Raw fundus images present significant challenges for deep learning models: they often contain large, uninformative black borders, suffer from inconsistent illumination, and feature low-contrast lesions that are difficult to detect.

This cell defines and executes a custom, multi-step image preprocessing pipeline designed to standardize the dataset and highlight the critical biomarkers of Diabetic Retinopathy (such as microaneurysms and exudates).

**Step-by-Step Breakdown of the Preprocessing Function:**

* **Automated Cropping (Contour Detection):** The function converts the image to grayscale, thresholds it, and identifies the largest external contour. It then computes a bounding box to crop away the excess black background, ensuring the model focuses exclusively on the retinal tissue.
* **Global Noise Reduction:** A Median Filter (`cv2.medianBlur`) is applied to smooth the image and eliminate salt-and-pepper noise without blurring critical edges.
* **Targeted Green Channel Enhancement:** Fundus images are composed of Red, Green, and Blue channels. In retinal imaging, the **green channel** provides the highest contrast for visualizing blood vessels, hemorrhages, and exudates, while the red channel is often oversaturated.
    * The channels are split, and the green channel is isolated.
    * A secondary Median filter is applied to the green channel specifically for localized noise reduction.
    * **CLAHE (Contrast Limited Adaptive Histogram Equalization)** is applied to the green channel. Unlike standard histogram equalization which operates globally, CLAHE computes histograms in small grid tiles, enhancing local contrast to make subtle lesions "pop" without over-amplifying noise.
* **Aspect-Ratio Preserving Resize & Padding:** Deep learning models require fixed input sizes (e.g., 224x224). Naively resizing a circular retina into a square distorts its shape, potentially warping lesions. This function calculates a scaling ratio to shrink the longest dimension to the target size, then pads the remaining space with black pixels, perfectly preserving the anatomical aspect ratio.

Finally, the script executes this function iteratively across the entire local dataset, writing the normalized, enhanced images back to the Colab disk (`PREPROCESSED_TRAIN_IMAGES_DIR`). By performing this heavily compute-intensive preprocessing **offline** (before training begins) rather than dynamically inside the data loader, we drastically accelerate the subsequent model training time.

In [ ]:
# --- Advanced Preprocessing Function (Offline Preprocessing) ---
def preprocess_image_advanced(image_path, target_size=(224, 224)):
    """
    Applies advanced preprocessing to a retinal image.
    Includes circle cropping, median filtering, CLAHE on the green channel, and Wiener filter.
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: Image not found or corrupted: {image_path}")
        return None

    # 1. Circle Cropping
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        img = img[y:y+h, x:x+w]

    # 2. Median Filter for noise reduction and smoothing
    img = cv2.medianBlur(img, 5)

    # 3. Green Channel processing
    b, g, r = cv2.split(img)

    # === Change: Wiener filter removed, using median filter for green channel noise reduction ===
    g_denoised = cv2.medianBlur(g, 5)

    # 4. Apply CLAHE on the green channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g_clahe = clahe.apply(g_denoised)

    # 5. Re-merge channels
    img_processed = cv2.merge([b, g_clahe, r])

    # 6. Smart Resizing maintaining Aspect Ratio
    h_orig, w_orig = img_processed.shape[:2]
    max_dim = max(h_orig, w_orig)
    ratio = target_size[0] / max_dim
    new_size = tuple(int(dim * ratio) for dim in [w_orig, h_orig])

    resized_img = cv2.resize(img_processed, new_size, interpolation=cv2.INTER_AREA)

    # 7. Add padding (black background) to reach target size (square)
    delta_w = target_size[0] - new_size[0]
    delta_h = target_size[1] - new_size[1]
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)

    padded_img = cv2.copyMakeBorder(resized_img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)

    return padded_img

# --- 4. Execute Preprocessing and Save Images ---
os.makedirs(PREPROCESSED_TRAIN_IMAGES_DIR, exist_ok=True)

# Preprocess training images
train_filenames = os.listdir(LOCAL_TRAIN_IMAGES_DIR)
print(f"\nStarting preprocessing of {len(train_filenames)} training images...")
for filename in train_filenames:
    image_path = os.path.join(LOCAL_TRAIN_IMAGES_DIR, filename)
    output_path = os.path.join(PREPROCESSED_TRAIN_IMAGES_DIR, filename)
    processed_img = preprocess_image_advanced(image_path)
    if processed_img is not None:
        cv2.imwrite(output_path, processed_img)

print("\nPreprocessing of all images completed.")
print(f"Preprocessed training images stored in: {PREPROCESSED_TRAIN_IMAGES_DIR}")


شروع پیش پردازش 3662 تصویر آموزشی...

پیش پردازش تمام تصاویر تکمیل شد.
تصاویر پیش پردازش شده آموزشی در: /content/aptos_preprocessed/preprocessed_train_images


##  Data Persistence and Backup

 all data stored on the local disk (`/content/`) is permanently deleted when the runtime disconnects or times out. Because the advanced image preprocessing pipeline we just executed is computationally expensive and time-consuming, it is essential to save the results to persistent storage.

This cell establishes a pipeline back to your connected Google Drive and uses the `rsync` utility to efficiently transfer the directory of fully preprocessed images.

By archiving this standardized dataset, future modeling sessions can bypass the initial raw data transfer and preprocessing phases entirely, loading the optimized images directly to begin model training immediately.

In [ ]:
# --- 1. Define Local and Google Drive Paths ---
# These paths should match the ones defined in your main code.
LOCAL_PREPROCESSED_TRAIN_IMAGES_DIR = PREPROCESSED_TRAIN_IMAGES_DIR

DESTINATION_DIR_ON_DRIVE = '/content/drive/MyDrive/Aptos/results'

# --- 2. Copy files from local storage to Google Drive ---
print(f"Copying final files from local storage to Google Drive at {DESTINATION_DIR_ON_DRIVE}...")

# Create destination folder on Google Drive if it doesn't exist
os.makedirs(DESTINATION_DIR_ON_DRIVE, exist_ok=True)

# Copy preprocessed training images folder
# Using rsync -avh which is efficient for copying folders.
!rsync -avh "$LOCAL_PREPROCESSED_TRAIN_IMAGES_DIR/" "$DESTINATION_DIR_ON_DRIVE/preprocessed_train_images/"

print("\nFile copy to Google Drive completed successfully.")
print("All your files are now saved in the Google Drive path.")

در حال کپی کردن فایل های نهایی از حافظه محلی به Google Drive در مسیر /content/drive/MyDrive/Aptos/results...
sending incremental file list
rsync: [sender] change_dir "/content/aptos_local_data/preprocessed_train_images" failed: No such file or directory (2)
created directory /content/drive/MyDrive/Aptos/results/preprocessed_train_images

sent 19 bytes  received 97 bytes  232.00 bytes/sec
total size is 0  speedup is 0.00
rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1338) [sender=3.2.7]

کپی فایل ها به Google Drive با موفقیت انجام شد.
تمام فایل های شما اکنون در مسیر Google Drive ذخیره شده اند.
